# Engagement Optimization Recommender

## Objective

To use historical social media performance patterns to recommend content
strategies based on category, content type, content length, and posting time.

The recommendations are based on observed engagement patterns and statistically
validated findings from the previous analysis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(
    "cleaned_social_media_engagement_dataset.csv",
    parse_dates=["Timestamp"]
)

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (5000, 22)


,Post_ID,Timestamp,Platform,Content_Type,Category,Likes,Comments,Shares,Views,Saves,...,Hour_of_Day,Day_of_Week,Hashtag_Count,Content_Length,Sentiment,Influencer_Tier,Has_Media,Is_Verified,Month,Total_Interactions
0,POST_04552,2024-01-01 01:42:00,Instagram,Carousel,Business,8287,247,51,29502,20,...,1,Monday,16,985,Positive,Macro,True,False,2024-01,8605
1,POST_02171,2024-01-01 05:05:00,LinkedIn,Document,Health,1711,27,247,24538,139,...,5,Monday,9,627,Negative,Macro,False,True,2024-01,2124
2,POST_00210,2024-01-01 09:18:00,Instagram,Carousel,Food,1527,191,7,5460,359,...,9,Monday,27,79,Positive,Macro,True,True,2024-01,2084
3,POST_01548,2024-01-01 10:58:00,Facebook,Video,Sports,535,178,433,68246,740,...,10,Monday,17,554,Neutral,Macro,True,False,2024-01,1886
4,POST_01350,2024-01-01 13:12:00,Instagram,Photo,Fitness,9706,35,118,25782,611,...,13,Monday,5,1136,Positive,Mid-tier,True,False,2024-01,10470


In [2]:
median_length = df["Content_Length"].median()

df["Length_Group"] = np.where(
    df["Content_Length"] <= median_length,
    "Short",
    "Long"
)

df["Time_Period"] = pd.cut(
    df["Hour_of_Day"],
    bins=[-1, 5, 11, 16, 20, 23],
    labels=[
        "Late Night",
        "Morning",
        "Afternoon",
        "Evening",
        "Night"
    ]
)

df[
    [
        "Content_Length",
        "Length_Group",
        "Hour_of_Day",
        "Time_Period"
    ]
].head()

,Content_Length,Length_Group,Hour_of_Day,Time_Period
0,985,Long,1,Late Night
1,627,Long,5,Late Night
2,79,Short,9,Morning
3,554,Long,10,Morning
4,1136,Long,13,Afternoon


In [3]:
category_performance = (
    df.groupby("Category")
      .agg(
          Posts=("Post_ID", "count"),
          Median_Engagement=("Engagement_Rate", "median"),
          Median_Shares=("Shares", "median"),
          Median_Saves=("Saves", "median")
      )
      .sort_values(
          "Median_Engagement",
          ascending=False
      )
)

category_performance.round(2)

,Posts,Median_Engagement,Median_Shares,Median_Saves
Category,,,,
Travel,398,2.64,461.5,420.5
Technology,434,2.60,410.5,425.0
Lifestyle,433,2.52,351.0,427.0
Entertainment,432,2.39,397.0,409.0
Education,413,2.37,471.0,442.0
Fitness,393,2.36,392.0,403.0
Sports,401,2.36,453.0,428.0
Food,413,2.30,482.0,420.0
Gaming,476,2.24,397.5,438.5


In [4]:
content_performance = (
    df.groupby("Content_Type")
      .agg(
          Posts=("Post_ID", "count"),
          Median_Engagement=("Engagement_Rate", "median"),
          Median_Shares=("Shares", "median"),
          Median_Saves=("Saves", "median")
      )
      .sort_values(
          "Median_Engagement",
          ascending=False
      )
)

content_performance.round(2)

,Posts,Median_Engagement,Median_Shares,Median_Saves
Content_Type,,,,
Duet,248,11.54,2497.0,1573.0
Stitch,259,10.68,2281.0,1677.0
Video,616,4.56,806.5,725.5
Short,121,3.95,1043.0,2449.0
Community Post,135,3.91,1038.0,2732.0
Live,358,2.41,619.0,550.5
Carousel,333,2.25,140.0,511.0
Photo,320,2.12,156.0,481.0
Reel,306,2.04,155.5,522.0


In [5]:
length_performance = (
    df.groupby("Length_Group")["Engagement_Rate"]
      .median()
      .sort_values(ascending=False)
)

time_performance = (
    df.groupby(
        "Time_Period",
        observed=True
    )["Engagement_Rate"]
      .median()
      .sort_values(ascending=False)
)

print("Content Length Performance:")
print(length_performance)

print("\nPosting Time Performance:")
print(time_performance)

Content Length Performance:
Length_Group
Short    3.33
Long     1.72
Name: Engagement_Rate, dtype: float64

Posting Time Performance:
Time_Period
Afternoon     2.520
Evening       2.375
Night         2.330
Morning       2.170
Late Night    2.110
Name: Engagement_Rate, dtype: float64


## Platform-Aware Recommendation Scoring

Content formats and engagement scales differ across platforms. Therefore,
categories and content types are evaluated relative to other options available
on the same platform.

The recommendation score considers:

- Median Engagement Rate: 30%
- Median Shares: 35%
- Median Saves: 35%

Shares and Saves receive greater weight because they represent stronger
high-value audience actions.

In [6]:
category_strategy = (
    df.groupby(["Platform", "Category"])
      .agg(
          Posts=("Post_ID", "count"),
          Median_Engagement=("Engagement_Rate", "median"),
          Median_Shares=("Shares", "median"),
          Median_Saves=("Saves", "median")
      )
      .reset_index()
)

category_strategy.head()

,Platform,Category,Posts,Median_Engagement,Median_Shares,Median_Saves
0,Facebook,Business,83,1.720,471.0,381.0
1,Facebook,Education,73,1.720,640.0,379.0
2,Facebook,Entertainment,86,1.975,501.5,278.0
3,Facebook,Fashion,90,2.030,567.5,445.5
4,Facebook,Fitness,75,1.940,497.0,393.0


In [7]:
for col in [
    "Median_Engagement",
    "Median_Shares",
    "Median_Saves"
]:
    category_strategy[f"{col}_Score"] = (
        category_strategy
        .groupby("Platform")[col]
        .rank(pct=True)
    )

category_strategy["Strategy_Score"] = (
    0.30 * category_strategy["Median_Engagement_Score"]
    + 0.35 * category_strategy["Median_Shares_Score"]
    + 0.35 * category_strategy["Median_Saves_Score"]
) * 100

category_strategy = category_strategy.sort_values(
    ["Platform", "Strategy_Score"],
    ascending=[True, False]
)

category_strategy[
    [
        "Platform",
        "Category",
        "Posts",
        "Median_Engagement",
        "Median_Shares",
        "Median_Saves",
        "Strategy_Score"
    ]
].head(20).round(2)


,Platform,Category,Posts,Median_Engagement,Median_Shares,Median_Saves,Strategy_Score
3,Facebook,Fashion,90,2.03,567.5,445.5,86.67
5,Facebook,Food,80,1.94,551.5,421.0,69.17
10,Facebook,Technology,85,2.52,507.0,393.0,68.12
8,Facebook,Lifestyle,83,2.52,349.0,457.0,66.67
9,Facebook,Sports,84,2.02,490.0,394.5,55.00
6,Facebook,Gaming,97,1.48,512.0,409.0,52.08
7,Facebook,Health,85,1.86,467.0,428.0,47.92
4,Facebook,Fitness,75,1.94,497.0,393.0,47.29
1,Facebook,Education,73,1.72,640.0,379.0,47.08
11,Facebook,Travel,63,2.30,462.0,391.0,42.50


In [8]:
content_strategy = (
    df.groupby(["Platform", "Content_Type"])
      .agg(
          Posts=("Post_ID", "count"),
          Median_Engagement=("Engagement_Rate", "median"),
          Median_Shares=("Shares", "median"),
          Median_Saves=("Saves", "median")
      )
      .reset_index()
)

for col in [
    "Median_Engagement",
    "Median_Shares",
    "Median_Saves"
]:
    content_strategy[f"{col}_Score"] = (
        content_strategy
        .groupby("Platform")[col]
        .rank(pct=True)
    )

content_strategy["Strategy_Score"] = (
    0.30 * content_strategy["Median_Engagement_Score"]
    + 0.35 * content_strategy["Median_Shares_Score"]
    + 0.35 * content_strategy["Median_Saves_Score"]
) * 100

content_strategy = content_strategy.sort_values(
    ["Platform", "Strategy_Score"],
    ascending=[True, False]
)

content_strategy[
    [
        "Platform",
        "Content_Type",
        "Posts",
        "Median_Engagement",
        "Median_Shares",
        "Median_Saves",
        "Strategy_Score"
    ]
].head(20).round(2)

,Platform,Content_Type,Posts,Median_Engagement,Median_Shares,Median_Saves,Strategy_Score
0,Facebook,Live,239,1.76,557.0,428.0,77.50
1,Facebook,Post,240,2.46,490.5,396.5,73.75
3,Facebook,Video,261,2.01,462.0,411.0,57.50
2,Facebook,Story,244,1.84,490.0,365.5,41.25
7,Instagram,Story,324,2.12,151.0,532.0,67.50
5,Instagram,Photo,320,2.12,156.0,481.0,66.25
6,Instagram,Reel,306,2.04,155.5,522.0,60.00
4,Instagram,Carousel,333,2.25,140.0,511.0,56.25
9,LinkedIn,Document,132,0.59,266.5,152.5,91.25
8,LinkedIn,Article,116,0.58,260.5,145.0,61.87


In [9]:
platform_time_performance = (
    df.groupby(
        ["Platform", "Time_Period"],
        observed=True
    )["Engagement_Rate"]
    .median()
    .reset_index()
)

platform_time_performance.head(10)

,Platform,Time_Period,Engagement_Rate
0,Facebook,Late Night,1.70
1,Facebook,Morning,2.16
2,Facebook,Afternoon,2.26
3,Facebook,Evening,1.86
4,Facebook,Night,2.05
5,Instagram,Late Night,2.11
6,Instagram,Morning,2.02
7,Instagram,Afternoon,2.26
8,Instagram,Evening,2.05
9,Instagram,Night,2.17


## Recommendation Function

The final recommender accepts a social media platform and returns the
historically strongest category and content format for that platform.

Short content receives a strong recommendation because its engagement
difference was statistically significant during A/B testing.

Posting time is provided only as a suggestion because differences across
time periods were not statistically significant.

In [10]:
def recommend_strategy(platform):

    platform_lookup = {
        p.lower(): p
        for p in df["Platform"].unique()
    }

    platform_key = platform.strip().lower()

    if platform_key not in platform_lookup:
        return {
            "Error": "Platform not found",
            "Available Platforms": sorted(df["Platform"].unique())
        }

    platform_name = platform_lookup[platform_key]

    # Best category
    category_rows = category_strategy[
        category_strategy["Platform"] == platform_name
    ]

    best_category = category_rows.iloc[0]

    # Best content type
    content_rows = content_strategy[
        content_strategy["Platform"] == platform_name
    ]

    best_content = content_rows.iloc[0]

    # Statistically supported length recommendation
    recommended_length = "Short"

    # Best observed time, but not statistically significant
    platform_times = platform_time_performance[
    platform_time_performance["Platform"] == platform_name
]

    suggested_time = (
        platform_times
        .sort_values("Engagement_Rate", ascending=False)
        .iloc[0]["Time_Period"]
    )

    return {
        "Platform": platform_name,

        "Recommended_Category":
            best_category["Category"],

        "Category_Strategy_Score":
            round(best_category["Strategy_Score"], 2),

        "Recommended_Content_Type":
            best_content["Content_Type"],

        "Content_Type_Strategy_Score":
            round(best_content["Strategy_Score"], 2),

        "Recommended_Length":
            recommended_length,

        "Suggested_Posting_Time":
            suggested_time,

        "Time_Note":
            "Suggested from historical median engagement; posting-time differences were not statistically significant."
    }

In [11]:
recommend_strategy("Instagram")

{'Platform': 'Instagram',
 'Recommended_Category': 'Sports',
 'Category_Strategy_Score': np.float64(91.67),
 'Recommended_Content_Type': 'Story',
 'Content_Type_Strategy_Score': np.float64(67.5),
 'Recommended_Length': 'Short',
 'Suggested_Posting_Time': 'Afternoon',
 'Time_Note': 'Suggested from historical median engagement; posting-time differences were not statistically significant.'}

In [12]:
recommend_strategy("TikTok")

{'Platform': 'TikTok',
 'Recommended_Category': 'Entertainment',
 'Category_Strategy_Score': np.float64(75.0),
 'Recommended_Content_Type': 'Duet',
 'Content_Type_Strategy_Score': np.float64(88.33),
 'Recommended_Length': 'Short',
 'Suggested_Posting_Time': 'Morning',
 'Time_Note': 'Suggested from historical median engagement; posting-time differences were not statistically significant.'}

## Final Platform Recommendations

The recommendation engine is applied to every platform in the dataset to
create a consolidated strategy table for dashboard and reporting purposes.

In [13]:
all_recommendations = []

for platform in sorted(df["Platform"].unique()):
    result = recommend_strategy(platform)
    all_recommendations.append(result)

recommendations_df = pd.DataFrame(all_recommendations)

recommendations_df

,Platform,Recommended_Category,Category_Strategy_Score,Recommended_Content_Type,Content_Type_Strategy_Score,Recommended_Length,Suggested_Posting_Time,Time_Note
0,Facebook,Fashion,86.67,Live,77.50,Short,Afternoon,Suggested from historical median engagement; p...
1,Instagram,Sports,91.67,Story,67.50,Short,Afternoon,Suggested from historical median engagement; p...
2,LinkedIn,Lifestyle,79.58,Document,91.25,Short,Evening,Suggested from historical median engagement; p...
3,TikTok,Entertainment,75.00,Duet,88.33,Short,Morning,Suggested from historical median engagement; p...
4,Twitter,Fitness,88.75,Poll,83.75,Short,Morning,Suggested from historical median engagement; p...
5,YouTube,Fashion,79.58,Short,76.25,Short,Evening,Suggested from historical median engagement; p...


## Recommendation Engine Summary

The recommendation engine provides platform-specific content strategies using
historical engagement, shares, and saves.

Content category and format recommendations are calculated relative to each
platform, while Short content is recommended based on the statistically
significant A/B testing result.

Posting time is treated only as a historical suggestion because differences
between posting-time groups were not statistically significant.

In [16]:
recommendations_df["Category_Strategy_Score"] = (
    recommendations_df["Category_Strategy_Score"]
    .astype(float)
)

recommendations_df["Content_Type_Strategy_Score"] = (
    recommendations_df["Content_Type_Strategy_Score"]
    .astype(float)
)

recommendations_df

,Platform,Recommended_Category,Category_Strategy_Score,Recommended_Content_Type,Content_Type_Strategy_Score,Recommended_Length,Suggested_Posting_Time,Time_Note
0,Facebook,Fashion,86.67,Live,77.50,Short,Afternoon,Suggested from historical median engagement; p...
1,Instagram,Sports,91.67,Story,67.50,Short,Afternoon,Suggested from historical median engagement; p...
2,LinkedIn,Lifestyle,79.58,Document,91.25,Short,Evening,Suggested from historical median engagement; p...
3,TikTok,Entertainment,75.00,Duet,88.33,Short,Morning,Suggested from historical median engagement; p...
4,Twitter,Fitness,88.75,Poll,83.75,Short,Morning,Suggested from historical median engagement; p...
5,YouTube,Fashion,79.58,Short,76.25,Short,Evening,Suggested from historical median engagement; p...


In [17]:
recommendations_df.to_csv(
    "platform_recommendations.csv",
    index=False
)

print("Platform recommendations saved successfully.")

Platform recommendations saved successfully.


In [18]:
import zipfile

files_to_zip = [
    "cleaned_social_media_engagement_dataset.csv",
    "cleaned_youtube_comments_nlp.csv",
    "platform_recommendations.csv"
]

with zipfile.ZipFile("powerbi_data.zip", "w") as zipf:
    for file in files_to_zip:
        zipf.write(file)

print("ZIP created successfully.")

ZIP created successfully.
